# Compositional OOD — B, F1 and F2

Audit and zero-shot inference only. Existing E1–E5 and official splits remain unchanged. No F3/F4. Default execution performs the metadata audit without downloading weights.

## Setup
Select the school environment kernel. Install the root requirements and the additional requirements described in README first.

In [ ]:
from pathlib import Path
import sys, json, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common/dataset.py").is_file())
sys.path.insert(0, str(ROOT))
from approach3_ood_backbones.experiments import audit, run
CONFIG = json.loads((ROOT / "approach3_ood_backbones/configs/default.json").read_text())
RUN_MODELS = False  # Enable only after weights and environment are ready.
ALLOW_DOWNLOAD = False  # Explicit permission for large model downloads.
FULL = False  # Keep False until the school GPU smoke passes.
print(torch.__version__, "CUDA:", torch.cuda.is_available())

## Data
Phase B: inspect the training-only compositional holdout. Current metadata gives 32 held-out pair classes: 6,119 reduced-training and 1,104 holdout images. Every crop/disease remains represented in reduced training. Original validation and tests are untouched.

In [ ]:
audit_path, audit_record = audit(CONFIG)
print(audit_path)
print(json.dumps(audit_record["split"], indent=2))

## Model
Four model-specific adapters: OpenAI CLIP B/16, CLIP L/14, Google SigLIP2 and BioCLIP. Each uses its own tokenizer and preprocessing; all predict the same 333 candidate rows. Model loading and a two-image inference preflight occur inside each runner before evaluation. Downloads remain disabled unless explicitly enabled.

In [ ]:
print(json.dumps(CONFIG["models"], indent=2))

## Training
No neural-weight training occurs in F1/F2. F2 performs validation-only coefficient selection using the additional pair-disjoint holdout, and seals the selection before official test images are scored. The designated reduced training partition is 6,119 images, but these zero-shot backbones optimize on zero images. No retraining on all 7,223 images is performed. F3/F4 are not implemented.

In [ ]:
print("Neural training images: 0")
print("F2 coefficient grid:", CONFIG["coefficient_grid"])

## Evaluation
F1 benchmarks the four zero-shot backbones. F2 selects coefficients on compositional validation, freezes them, then evaluates pair-only and factorized scoring. Set RUN_MODELS=True only after environment and weight checks. Smoke results are mechanics checks, not formal accuracy evidence.

In [ ]:
if RUN_MODELS:
    f1_path, f1_reports = run(CONFIG, "F1", ["clip_b16", "clip_l14", "siglip2", "bioclip"], smoke=not FULL, allow_download=ALLOW_DOWNLOAD, robust=FULL)
    print(f1_path)
else:
    print("F1 skipped: RUN_MODELS=False; no weights downloaded.")

### F2 — Factorized scoring
Compare raw pair-only, calibrated pair-only and calibrated factorized scores. The runner writes the selection record before official test inference.

In [ ]:
if RUN_MODELS:
    f2_path, f2_reports = run(CONFIG, "F2", ["siglip2"], smoke=not FULL, allow_download=ALLOW_DOWNLOAD, robust=FULL)
    print(f2_path)
else:
    print("F2 skipped: RUN_MODELS=False; no weights downloaded.")

## Save
Every run already saves comparison.csv/json, ood_breakdown.csv, split manifests and confusion files in a unique directory under results/ood_backbones/. Empty component categories are N/A. No trained checkpoints exist for these zero-shot experiments. Full GPU inference remains to be validated at school. F3/F4 require separate approval.